# GWAS + Random Forest Pipeline
## Gallstone Disease SNP Analysis

**Workflow:**
1. Load professor's top-SNP list (`summary_results_sorted_by_p.csv`)
2. Load per-gene `.raw` files — only the top-SNP columns
3. Load phenotype from `Merged file with metabolites.xlsx`
4. Merge on IID
5. EDA plots
6. Single-SNP association statistics
7. Random Forest classification
8. Results summary

In [1]:
# ============================================================
# CELL 1 — CONFIGURATION
# No need to change anything — paths are auto-detected!
# Just make sure this notebook is inside your project_data folder
# ============================================================

import os
from pathlib import Path

# Auto-detect: use the folder where this notebook lives
# Works whether you run it from VS Code, Jupyter, or anywhere else
try:
    # When running as a notebook, __file__ is not defined
    # so we use the current working directory instead
    BASE = Path(os.getcwd())
except:
    BASE = Path('.')

print(f'Project folder detected as: {BASE}')

CONFIG = {
    'snp_summary_csv' : BASE / 'summary_results sorted by p (1).csv',
    'raw_files_dir'   : BASE,
    'phenotype_xlsx'  : BASE / 'Merged file with metabolites.xlsx',

    # Column names in phenotype file
    'iid_col'         : 'IID',
    'target_col'      : 'gall',

    # Keep SNPs with GWAS p-value <= this threshold
    # Set to 1.0 to include ALL SNPs
    'p_value_cutoff'  : 1e-5,

    # Random Forest settings
    'n_estimators'    : 500,
    'max_depth'       : 8,
    'test_size'       : 0.20,
    'cv_folds'        : 5,
    'random_state'    : 42,

    'run_assoc_stats' : True,
    'output_dir'      : BASE / 'gwas_output',
}

# ── Verify all files exist before running ──────────────────
print('\nChecking files...')
checks = {
    'SNP summary CSV'       : CONFIG['snp_summary_csv'],
    'Phenotype xlsx'        : CONFIG['phenotype_xlsx'],
    'project_data folder'   : CONFIG['raw_files_dir'],
}
all_ok = True
for name, path in checks.items():
    found = Path(path).exists()
    icon  = '✅' if found else '❌ NOT FOUND'
    print(f'  {icon}  {name}')
    print(f'       {path}')
    if not found:
        all_ok = False

# Count .raw files
raw_files = list(Path(CONFIG['raw_files_dir']).glob('*_region.raw'))
print(f'\n  ✅  .raw files found: {len(raw_files)}')
for r in raw_files:
    print(f'       {r.name}')

if all_ok and len(raw_files) > 0:
    print('\n✅ Everything found — proceed to Cell 2!')
else:
    print('\n❌ Some files missing.')
    print('   Make sure this notebook is saved inside project_data folder,')
    print('   then restart the kernel and run again.')


Project folder detected as: c:\Users\bsevak\Documents\project_data

Checking files...
  ✅  SNP summary CSV
       c:\Users\bsevak\Documents\project_data\summary_results sorted by p (1).csv
  ✅  Phenotype xlsx
       c:\Users\bsevak\Documents\project_data\Merged file with metabolites.xlsx
  ✅  project_data folder
       c:\Users\bsevak\Documents\project_data

  ✅  .raw files found: 18
       CDHR3_region.raw
       CLDN7_region.raw
       GPR78_region.raw
       GPRIN3_region.raw
       HLA-G_region.raw
       HMGB1P5_region.raw
       LINC01122_region.raw
       LRP8_region.raw
       LUZP2_region.raw
       MATN2_region.raw
       OSBPL10_region.raw
       PALM2AKAP2_region.raw
       PTPRN2_region.raw
       RNFT2_region.raw
       RPL31P35_region.raw
       SEC61G_region.raw
       TJP1_region.raw
       UGGT1_region.raw

✅ Everything found — proceed to Cell 2!


In [2]:
pip install pandas numpy scipy scikit-learn matplotlib seaborn openpyxl

Note: you may need to restart the kernel to use updated packages.


In [3]:
# ============================================================
# CELL 2 — IMPORTS
# Run: pip install pandas numpy scipy scikit-learn matplotlib seaborn openpyxl
# ============================================================

import warnings, logging
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    roc_auc_score, roc_curve, accuracy_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)
from matplotlib.patches import Patch

warnings.filterwarnings('ignore')
%matplotlib inline
plt.rcParams['figure.dpi'] = 120

# Create output folder
OUT = Path(CONFIG['output_dir'])
OUT.mkdir(exist_ok=True)

print('All imports OK. Output folder:', OUT.resolve())

All imports OK. Output folder: C:\Users\bsevak\Documents\project_data\gwas_output


In [4]:
# ============================================================
# CELL 3 — LOAD TOP-SNP SUMMARY FILE
# Columns: Gene, Metabolite, TopSNP, MinP
# SNP format: chr:pos:REF:ALT  e.g. 7:104129994:T:G
# ============================================================

snp_raw = pd.read_csv(CONFIG['snp_summary_csv'])
print(f'Loaded {len(snp_raw):,} rows from SNP summary')
print('Columns:', list(snp_raw.columns))
display(snp_raw.head(5))

# Filter by p-value cutoff
snp_raw = snp_raw[snp_raw['MinP'] <= CONFIG['p_value_cutoff']].copy()
print(f'\nAfter p <= {CONFIG["p_value_cutoff"]:.0e}: {len(snp_raw):,} rows')

# Deduplicate: same SNP may appear multiple times (different metabolites)
snp_df = (snp_raw[['Gene', 'TopSNP', 'MinP']]
          .sort_values('MinP')
          .drop_duplicates(subset='TopSNP')
          .reset_index(drop=True))

print(f'Unique SNPs : {len(snp_df):,}')
print(f'Genes found : {sorted(snp_df["Gene"].unique())}')
display(snp_df.head(10))

Loaded 9,168 rows from SNP summary
Columns: ['Gene', 'Metabolite', 'TopSNP', 'MinP']


,Gene,Metabolite,TopSNP,MinP
0,CLDN7,Propranolol,17:8982569:C:G,1.000000e-08
1,MATN2,Uracil,8:98680059:TA:T,8.500000e-08
2,LRP8,LRP8_reg_Glyceric_acid,1:52165225:A:G,1.070000e-07
3,UGGT1,Adonitol,2:127540374:A:G,1.350000e-07
4,GPC6,GPC6_reg_L_Arabitol,13:95769509:TA:T,1.430000e-07



After p <= 1e-05: 592 rows
Unique SNPs : 479
Genes found : ['CDHR3', 'CLDN7', 'CPZ', 'ERICH1', 'EXOC4', 'GALNTL6', 'GPC6', 'GPRIN3', 'HLA-G', 'HMGB1P5', 'LINC0112', 'LRP8', 'LUZP2', 'MATN2', 'OSBPL10', 'PALM2AKA', 'PTPRN2', 'RNFT2', 'RPL31P35', 'TJP1', 'UGGT1']


,Gene,TopSNP,MinP
0,CLDN7,17:8982569:C:G,1.000000e-08
1,MATN2,8:98680059:TA:T,8.500000e-08
2,LRP8,1:52165225:A:G,1.070000e-07
3,UGGT1,2:127540374:A:G,1.350000e-07
4,GPC6,13:95769509:TA:T,1.430000e-07
5,CDHR3,7:104129994:T:G,1.780000e-07
6,CDHR3,7:105934803:G:A,1.970000e-07
7,LRP8,1:54194992:T:C,2.140000e-07
8,GALNTL6,4:174696434:A:AT,2.230000e-07
9,OSBPL10,3:30433384:A:AAT,2.380000e-07


In [5]:
# ============================================================
# CELL 4 — DISCOVER *_region.raw FILES
# ============================================================

raw_dir = Path(CONFIG['raw_files_dir'])
raw_files = sorted(set(list(raw_dir.glob('*_region.raw')) +
                        list(raw_dir.glob('*region.raw'))))

if not raw_files:
    raise FileNotFoundError(
        f'No *_region.raw files found in: {raw_dir.resolve()}\n'
        'Set raw_files_dir in CONFIG to the correct folder.'
    )

# Build gene -> path mapping
gene_map = {}
for f in raw_files:
    gene = f.stem.replace('_region', '').replace('region', '').strip('_')
    gene_map[gene] = f
    print(f'  {f.name:35s} -> gene key: {gene}')

gene_map_upper = {k.upper(): v for k, v in gene_map.items()}
print(f'\nTotal .raw files found: {len(gene_map)}')

  CDHR3_region.raw                    -> gene key: CDHR3
  CLDN7_region.raw                    -> gene key: CLDN7
  GPR78_region.raw                    -> gene key: GPR78
  GPRIN3_region.raw                   -> gene key: GPRIN3
  HLA-G_region.raw                    -> gene key: HLA-G
  HMGB1P5_region.raw                  -> gene key: HMGB1P5
  LINC01122_region.raw                -> gene key: LINC01122
  LRP8_region.raw                     -> gene key: LRP8
  LUZP2_region.raw                    -> gene key: LUZP2
  MATN2_region.raw                    -> gene key: MATN2
  OSBPL10_region.raw                  -> gene key: OSBPL10
  PALM2AKAP2_region.raw               -> gene key: PALM2AKAP2
  PTPRN2_region.raw                   -> gene key: PTPRN2
  RNFT2_region.raw                    -> gene key: RNFT2
  RPL31P35_region.raw                 -> gene key: RPL31P35
  SEC61G_region.raw                   -> gene key: SEC61G
  TJP1_region.raw                     -> gene key: TJP1
  UGGT1_region

In [6]:
# ============================================================
# CELL 5 — LOAD GENOTYPE DATA (top SNPs only)
#
# PLINK .raw SNP columns look like:  7:104129994:T:G_G
# We strip the allele suffix (_G) to get: 7:104129994:T:G
# ============================================================

def load_raw_for_gene(raw_path, snps_for_gene):
    """Load one .raw file, keeping only IID + requested SNP columns."""
    header = pd.read_csv(raw_path, sep=r'\s+', nrows=0)
    all_cols = list(header.columns)
    snp_set  = set(snps_for_gene)

    # Map raw_col -> base snp id
    col_map = {}
    for c in all_cols:
        base = c.rsplit('_', 1)[0]   # strip allele suffix
        if base in snp_set:
            col_map[c] = base

    found     = list(col_map.keys())
    not_found = snp_set - set(col_map.values())
    if not_found:
        print(f'  WARNING: {len(not_found)} SNP(s) not found in {raw_path.name}')
    if not found:
        print(f'  SKIP: no matching SNPs in {raw_path.name}')
        return pd.DataFrame()

    df = pd.read_csv(raw_path, sep=r'\s+', usecols=['IID'] + found, low_memory=False)
    df.rename(columns=col_map, inplace=True)
    df['IID'] = df['IID'].astype(str)
    print(f'  {raw_path.name}: {len(df):,} samples, {len(found)} SNPs loaded')
    return df


# Group SNPs by gene, load matching .raw file for each
gene_snps = snp_df.groupby('Gene')['TopSNP'].apply(list).to_dict()
frames = []

for gene, snps in gene_snps.items():
    g_up     = gene.upper().strip()
    raw_path = gene_map_upper.get(g_up)

    if raw_path is None:
        # Fuzzy match: prefix overlap
        matches = [k for k in gene_map_upper
                   if k.startswith(g_up) or g_up.startswith(k)]
        if matches:
            raw_path = gene_map_upper[matches[0]]
            print(f'Fuzzy match: "{gene}" -> "{matches[0]}"')
        else:
            print(f'WARNING: No .raw file for gene "{gene}" - skipping')
            continue

    chunk = load_raw_for_gene(raw_path, snps)
    if not chunk.empty:
        frames.append(chunk)

# Merge all gene chunks on IID
print('\nMerging gene chunks on IID...')
geno = frames[0]
for f in frames[1:]:
    geno = pd.merge(geno, f, on='IID', how='outer')

# Remove any duplicate SNP columns
dupes = geno.columns[geno.columns.duplicated()].tolist()
if dupes:
    print(f'Removing {len(dupes)} duplicate columns')
    geno = geno.loc[:, ~geno.columns.duplicated()]

print(f'\nFinal genotype shape: {geno.shape}')
display(geno.iloc[:3, :6])

  CDHR3_region.raw: 3,700 samples, 36 SNPs loaded
  CLDN7_region.raw: 3,700 samples, 41 SNPs loaded


ParserError: Error tokenizing data. C error: Calling read(nbytes) on source failed. Try engine='python'.

In [ ]:
# ============================================================
# CELL 6 — LOAD PHENOTYPE DATA
# File: Merged file with metabolites.xlsx
# Needs columns: IID, gall (Yes/No)
# ============================================================

pheno_raw = pd.read_excel(CONFIG['phenotype_xlsx'], dtype=str)
pheno_raw = pheno_raw.applymap(lambda x: x.strip() if isinstance(x, str) else x)
print(f'Raw phenotype shape: {pheno_raw.shape}')
print('Columns:', list(pheno_raw.columns[:10]), '...')
display(pheno_raw.head(3))

iid_col    = CONFIG['iid_col']
target_col = CONFIG['target_col']

# Case-insensitive column matching
for col in [iid_col, target_col]:
    if col not in pheno_raw.columns:
        match = [c for c in pheno_raw.columns if c.lower() == col.lower()]
        if match:
            pheno_raw.rename(columns={match[0]: col}, inplace=True)
            print(f'Renamed column "{match[0]}" -> "{col}"')
        else:
            raise ValueError(f'Column "{col}" not found. Available: {list(pheno_raw.columns)}')

# Drop rows with missing IID
before = len(pheno_raw)
pheno_raw = pheno_raw[pheno_raw[iid_col].notna() & (pheno_raw[iid_col].str.strip() != '')]
print(f'Valid IID rows: {len(pheno_raw):,}  (dropped {before - len(pheno_raw):,})')

# Encode gall -> binary label
pheno_raw[target_col] = pheno_raw[target_col].str.lower()
pheno_raw = pheno_raw[pheno_raw[target_col].isin({'yes','no'})].copy()
pheno_raw['label'] = (pheno_raw[target_col] == 'yes').astype(int)

pheno = pheno_raw[[iid_col, target_col, 'label']].copy()
pheno[iid_col] = pheno[iid_col].astype(str)

print(f'Cases  (gall=Yes) : {pheno["label"].sum():,}')
print(f'Controls (gall=No): {(pheno["label"]==0).sum():,}')

In [ ]:
# ============================================================
# CELL 7 — MERGE GENOTYPE + PHENOTYPE ON IID
# ============================================================

print('Genotype IID sample :', geno['IID'].iloc[:3].tolist())
print('Phenotype IID sample:', pheno[iid_col].iloc[:3].tolist())

merged = pd.merge(geno, pheno, left_on='IID', right_on=iid_col, how='inner')

if len(merged) == 0:
    print('ERROR: Merge returned 0 rows!')
    print('IID values do not match between .raw files and phenotype xlsx.')
    print('Geno IIDs  (first 5):', geno['IID'].tolist()[:5])
    print('Pheno IIDs (first 5):', pheno[iid_col].tolist()[:5])
else:
    snp_cols = [c for c in merged.columns if c in set(geno.columns) - {'IID'}]
    print(f'Merged shape : {merged.shape}')
    print(f'SNP features : {len(snp_cols)}')
    print(f'Cases        : {merged["label"].sum():,}')
    print(f'Controls     : {(merged["label"]==0).sum():,}')
    merged.to_csv(OUT / 'merged_dataset.csv', index=False)
    print(f'Saved: {OUT}/merged_dataset.csv')
    display(merged.iloc[:3, :8])

In [ ]:
# ============================================================
# CELL 8 — EXPLORATORY DATA ANALYSIS
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# --- Plot 1: Case/Control distribution ---
ax = axes[0]
counts = merged[target_col].value_counts()
bars = ax.bar(
    ['Control\n(gall=No)', 'Case\n(gall=Yes)'],
    [counts.get('no', 0), counts.get('yes', 0)],
    color=['#3498db', '#e74c3c'], edgecolor='white', width=0.5
)
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
            str(int(bar.get_height())), ha='center', fontweight='bold', fontsize=12)
ax.set_title('Case / Control Distribution', fontsize=13)
ax.set_ylabel('Count')
ax.spines[['top','right']].set_visible(False)

# --- Plot 2: SNP missing rate ---
ax2 = axes[1]
snp_num = merged[snp_cols].apply(pd.to_numeric, errors='coerce')
miss_pct = snp_num.isnull().mean() * 100
miss_nonzero = miss_pct[miss_pct > 0].sort_values(ascending=False)
if len(miss_nonzero) > 0:
    ax2.bar(range(len(miss_nonzero)), miss_nonzero.values, color='#e67e22')
    ax2.set_xticks(range(len(miss_nonzero)))
    ax2.set_xticklabels(miss_nonzero.index, rotation=90, fontsize=6)
    ax2.set_ylabel('% Missing')
    ax2.set_title('SNP Missing Data Rate')
else:
    ax2.text(0.5, 0.5, 'No missing SNP data', ha='center', va='center',
             transform=ax2.transAxes, fontsize=12)
    ax2.set_title('SNP Missing Data Rate')

plt.tight_layout()
plt.savefig(OUT / 'eda_distribution_missing.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: eda_distribution_missing.png')

In [ ]:
# ============================================================
# CELL 9 — SNP CORRELATION HEATMAP (top 50 SNPs)
# ============================================================

plot_cols = snp_cols[:50]
snp_num_plot = merged[plot_cols].apply(pd.to_numeric, errors='coerce')

if len(plot_cols) >= 2:
    fig, ax = plt.subplots(figsize=(14, 12))
    corr = snp_num_plot.corr()
    mask = np.triu(np.ones_like(corr, dtype=bool))
    sns.heatmap(corr, mask=mask, cmap='coolwarm', center=0,
                square=True, linewidths=0.2, ax=ax,
                cbar_kws={'shrink': 0.7})
    ax.set_xticklabels(ax.get_xticklabels(), fontsize=5, rotation=90)
    ax.set_yticklabels(ax.get_yticklabels(), fontsize=5)
    ax.set_title(f'SNP Correlation Matrix (top {len(plot_cols)} SNPs)', fontsize=12)
    plt.tight_layout()
    plt.savefig(OUT / 'correlation_heatmap.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved: correlation_heatmap.png')

In [ ]:
# ============================================================
# CELL 10 — SINGLE-SNP ASSOCIATION STATISTICS
# Chi-square or Fisher's exact test per SNP
# ============================================================

snp_info = snp_df.set_index('TopSNP')[['Gene','MinP']].to_dict('index')
records  = []

for snp in snp_cols:
    col   = pd.to_numeric(merged[snp], errors='coerce')
    y     = merged['label']
    valid = col.notna() & y.notna()
    cv, yv = col[valid], y[valid]
    if len(cv) < 10:
        continue
    try:
        table = pd.crosstab(cv.astype(int), yv)
        if table.shape == (2, 2):
            odds, pval = stats.fisher_exact(table)
            test = 'Fisher'
        else:
            chi2, pval, _, _ = stats.chi2_contingency(table)
            odds = np.nan
            test = 'Chi-square'
    except Exception:
        pval, odds, test = np.nan, np.nan, 'error'

    info = snp_info.get(snp, {})
    records.append({
        'SNP'        : snp,
        'Gene'       : info.get('Gene', ''),
        'GWAS_MinP'  : info.get('MinP', np.nan),
        'Assoc_Test' : test,
        'p_value'    : pval,
        'OddsRatio'  : odds,
        'FreqCase'   : round(cv[yv==1].mean()/2, 4),
        'FreqControl': round(cv[yv==0].mean()/2, 4),
        'N_case'     : int((yv==1).sum()),
        'N_control'  : int((yv==0).sum()),
    })

assoc_df = pd.DataFrame(records).sort_values('p_value').reset_index(drop=True)
assoc_df['-log10p'] = -np.log10(assoc_df['p_value'].clip(lower=1e-300))
assoc_df.to_csv(OUT / 'association_stats.csv', index=False)

print(f'Association stats computed for {len(assoc_df)} SNPs')
print(f'  SNPs p < 0.05 : {(assoc_df["p_value"] < 0.05).sum()}')
print(f'  SNPs p < 5e-8 : {(assoc_df["p_value"] < 5e-8).sum()}')
print('Saved: association_stats.csv')
display(assoc_df.head(10))

In [ ]:
# ============================================================
# CELL 11 — MINI-MANHATTAN PLOT (coloured by gene)
# ============================================================

genes   = assoc_df['Gene'].unique()
palette = plt.cm.tab20.colors
gene_color = {g: palette[i % len(palette)] for i, g in enumerate(genes)}
colors     = [gene_color[g] for g in assoc_df['Gene']]

fig, ax = plt.subplots(figsize=(max(12, len(assoc_df)*0.15), 5))
ax.bar(range(len(assoc_df)), assoc_df['-log10p'], color=colors, width=0.8)
ax.axhline(-np.log10(0.05), color='orange', linestyle='--',
           linewidth=1.2, label='p = 0.05')
ax.axhline(-np.log10(5e-8), color='red', linestyle='--',
           linewidth=1.2, label='p = 5e-8 (GWAS threshold)')
ax.set_ylabel('-log\u2081\u2080(p)', fontsize=12)
ax.set_title('Single-SNP Association p-values', fontsize=13)
ax.set_xticks([])
ax.spines[['top','right']].set_visible(False)

handles = [Patch(color=gene_color[g], label=g) for g in genes]
handles += [
    plt.Line2D([0],[0], color='orange', linestyle='--', label='p=0.05'),
    plt.Line2D([0],[0], color='red',    linestyle='--', label='p=5e-8'),
]
ax.legend(handles=handles, fontsize=8, ncol=5, loc='upper right')
plt.tight_layout()
plt.savefig(OUT / 'manhattan_mini.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: manhattan_mini.png')

In [ ]:
# ============================================================
# CELL 12 — RANDOM FOREST CLASSIFICATION
# ============================================================

# Prepare feature matrix
X_raw = merged[snp_cols].apply(pd.to_numeric, errors='coerce')
y     = merged['label'].values
X     = SimpleImputer(strategy='median').fit_transform(X_raw)

print(f'Feature matrix : {X.shape}')
print(f'Cases          : {y.sum()}   Controls: {(y==0).sum()}')

# Train / test split
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y,
    test_size    = CONFIG['test_size'],
    random_state = CONFIG['random_state'],
    stratify     = y
)
print(f'Train: {len(X_tr)}   Test: {len(X_te)}')

# Train model
rf = RandomForestClassifier(
    n_estimators = CONFIG['n_estimators'],
    max_depth    = CONFIG['max_depth'],
    class_weight = 'balanced',
    random_state = CONFIG['random_state'],
    n_jobs       = -1
)

# Cross-validation on training set
cv = StratifiedKFold(n_splits=CONFIG['cv_folds'], shuffle=True,
                     random_state=CONFIG['random_state'])
cv_aucs = cross_val_score(rf, X_tr, y_tr, cv=cv, scoring='roc_auc', n_jobs=-1)
print(f'{CONFIG["cv_folds"]}-fold CV AUC: {cv_aucs.mean():.4f} +/- {cv_aucs.std():.4f}')

# Final fit on full training set
rf.fit(X_tr, y_tr)
y_pred = rf.predict(X_te)
y_prob = rf.predict_proba(X_te)[:, 1]

auc = roc_auc_score(y_te, y_prob)
acc = accuracy_score(y_te, y_pred)
print(f'\nTest AUC      : {auc:.4f}')
print(f'Test Accuracy : {acc:.4f}')
print('\nClassification Report:')
print(classification_report(y_te, y_pred, target_names=['Control','Case']))

In [ ]:
# ============================================================
# CELL 13 — ROC CURVE + CONFUSION MATRIX
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# ROC curve
fpr, tpr, _ = roc_curve(y_te, y_prob)
ax = axes[0]
ax.plot(fpr, tpr, color='#e74c3c', linewidth=2.5,
        label=f'Random Forest  AUC = {auc:.4f}')
ax.plot([0,1],[0,1], 'k--', linewidth=1)
ax.fill_between(fpr, tpr, alpha=0.08, color='#e74c3c')
ax.set_xlabel('False Positive Rate', fontsize=11)
ax.set_ylabel('True Positive Rate', fontsize=11)
ax.set_title('ROC Curve', fontsize=13)
ax.legend(fontsize=10)
ax.spines[['top','right']].set_visible(False)

# Confusion matrix
ax2 = axes[1]
ConfusionMatrixDisplay(
    confusion_matrix(y_te, y_pred),
    display_labels=['Control','Case']
).plot(ax=ax2, colorbar=False, cmap='Blues')
ax2.set_title('Confusion Matrix', fontsize=13)

plt.tight_layout()
plt.savefig(OUT / 'rf_roc_cm.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: rf_roc_cm.png')

In [ ]:
# ============================================================
# CELL 14 — FEATURE IMPORTANCE (top 30 SNPs)
# ============================================================

imp   = rf.feature_importances_
top_n = min(30, len(snp_cols))
idx   = np.argsort(imp)[::-1][:top_n]

fig, ax = plt.subplots(figsize=(10, 7))
bar_colors = plt.cm.RdYlGn(np.linspace(0.3, 0.9, top_n))[::-1]
ax.barh(range(top_n), imp[idx][::-1], color=bar_colors)
ax.set_yticks(range(top_n))
ax.set_yticklabels([snp_cols[i] for i in idx][::-1], fontsize=8)
ax.set_xlabel('Feature Importance (Gini)', fontsize=11)
ax.set_title(f'Top {top_n} SNP Feature Importances — Random Forest', fontsize=12)
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig(OUT / 'rf_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: rf_feature_importance.png')

# Table of top SNPs with gene info
imp_df = pd.DataFrame({
    'SNP'       : [snp_cols[i] for i in idx],
    'Importance': imp[idx].round(5)
})
imp_df = imp_df.merge(
    snp_df[['TopSNP','Gene','MinP']].rename(columns={'TopSNP':'SNP'}),
    on='SNP', how='left'
)
display(imp_df)

In [ ]:
# ============================================================
# CELL 15 — RESULTS SUMMARY
# ============================================================

results = {
    'Model'        : 'Random Forest',
    'Test_AUC'     : round(auc, 4),
    'Test_Accuracy': round(acc, 4),
    'CV_AUC_mean'  : round(cv_aucs.mean(), 4),
    'CV_AUC_std'   : round(cv_aucs.std(), 4),
    'N_SNP_features': len(snp_cols),
    'N_train'      : len(X_tr),
    'N_test'       : len(X_te),
    'n_estimators' : CONFIG['n_estimators'],
    'max_depth'    : CONFIG['max_depth'],
}

pd.DataFrame([results]).to_csv(OUT / 'results_summary.csv', index=False)

print('=' * 55)
print('  RESULTS SUMMARY')
print('=' * 55)
for k, v in results.items():
    print(f'  {k:<22}: {v}')
print('=' * 55)
print(f'\nAll outputs saved to: {OUT.resolve()}')
print('Files:')
for f in sorted(OUT.iterdir()):
    print(f'  {f.name}')